In [2]:
%reset -f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, gc, time, zipfile
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
# 将上一级目录添加到搜索路径中
import sys
sys.path.append(os.path.abspath(".."))
from evaluator import evaluate_all  # 确保导入了你上传的评估文件

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, accuracy_score, brier_score_loss, recall_score, precision_score)
from scipy import stats
import numpy as np

# 预测模型
from sklearn.linear_model import LogisticRegression    # Logit模型
from sklearn.ensemble import RandomForestClassifier     # 随机森林
from xgboost import XGBClassifier                       # XGBoost

# 评估与预处理
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler        # Logit 必须进行标准化
from sklearn.metrics import accuracy_score
from evaluator import evaluate_all

In [3]:
import joblib
file_path = "/root/autodl-tmp/DATA/data_bundles.pkl"
data_bundles = joblib.load(file_path)
main_cols = ['insider_trading', 'Stkcd', 'Trddt']
features = [col for col in data_bundles['set1']['train'].columns if col not in main_cols]

In [6]:
# ============================================================
# RF 版：SHAP + 自定义图（bar + beeswarm）+ 批量并行调度+ TopN 变量频率统计（跨 set×ratio 汇总）
#   - compute_shap / model_path 命名规则为 RF
# ============================================================
import os
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
np.random.seed(42)
# -----------------------------
# 0) 强制数值化
# -----------------------------
def force_numeric_float32(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == "object":
            s = (
                df[c].astype(str)
                .str.replace("[", "", regex=False)
                .str.replace("]", "", regex=False)
                .str.strip()
            )
            df[c] = pd.to_numeric(s, errors="coerce")
    df = df.apply(pd.to_numeric, errors="coerce")
    return df.astype(np.float32)

# -----------------------------
# 1) 画图函数（bar + beeswarm）
# -----------------------------
def draw_shap_bar_beeswarm_onefig(
    shap_values: np.ndarray,
    X_shap: pd.DataFrame,
    feature_name_map: dict | None = None,
    top_n: int = 10,
    figsize=(15, 4),
    dpi: int = 300,
    bar_color: str = "lightpink",
    bar_alpha: float = 0.5,
    out_path: str | None = None,
    transparent: bool = True,
):
    """
    给定 shap_values + X_shap → 画 1 张 bar + beeswarm 并保存。
    不做 SHAP 计算，不做批量，不做并行。
    返回：shap_df（包含 feature 与 mean_shap 排序）
    """
    if feature_name_map is None:
        feature_name_map = {}

    if not isinstance(X_shap, pd.DataFrame):
        X_shap = pd.DataFrame(X_shap)

    if shap_values.shape[1] != X_shap.shape[1]:
        raise ValueError(
            f"shap_values n_features={shap_values.shape[1]} != X_shap n_features={X_shap.shape[1]}"
        )

    feature_names = list(X_shap.columns)

    # 1) 全局 mean(|SHAP|) 排序
    mean_shap = np.abs(shap_values).mean(axis=0)
    shap_df = (
        pd.DataFrame({"feature": feature_names, "mean_shap": mean_shap})
        .sort_values("mean_shap", ascending=False)
        .reset_index(drop=True)
    )

    mean_shap_total = shap_df["mean_shap"].sum()
    if mean_shap_total == 0:
        mean_shap_total = 1.0

    # 2) TopN + 重排
    top_n = min(int(top_n), len(feature_names))
    sorted_features = shap_df["feature"].values[:top_n]
    mean_shap_sorted = shap_df["mean_shap"].values[:top_n]
    mean_shap_percent = mean_shap_sorted / mean_shap_total * 100

    orig_index = [feature_names.index(f) for f in sorted_features]
    shap_values_sorted = shap_values[:, orig_index]
    X_sorted = X_shap[list(sorted_features)]

    n_feat = top_n
    y_pos = np.arange(n_feat)[::-1]

    # 3) 画图：底层 bar + 上层 beeswarm
    fig = plt.figure(figsize=figsize, dpi=dpi)
    ax_sw = fig.add_axes([0.32, 0.11, 0.59, 0.77])
    ax_bar = ax_sw.twiny()

    ax_bar.set_zorder(0)
    ax_sw.set_zorder(1)
    ax_sw.patch.set_alpha(0)

    ax_bar.barh(
        y=y_pos,
        width=mean_shap_sorted,
        height=0.6,
        color=bar_color,
        alpha=bar_alpha,
        edgecolor="none",
        zorder=0,
    )

    xlim_bar = (mean_shap_sorted.max() * 1.05) if len(mean_shap_sorted) else 1.0
    ax_bar.set_xlim(0, xlim_bar)
    xticks_bar = np.linspace(0, xlim_bar, 10)
    ax_bar.set_xticks(xticks_bar)
    ax_bar.set_xticklabels([f"{x:.4f}" for x in xticks_bar])
    ax_bar.set_xlabel("Mean (|SHAP| value)", fontsize=7)
    ax_bar.set_yticks(y_pos)

    # beeswarm
    expl = shap.Explanation(
        values=shap_values_sorted,
        data=X_sorted.values,
        feature_names=list(sorted_features),
    )
    plt.sca(ax_sw)
    shap.plots.beeswarm(expl, show=False, plot_size=None)

    shap_lim = (np.abs(shap_values_sorted).max() * 1.2) if shap_values_sorted.size else 1.0
    ax_sw.set_xlim(-shap_lim, shap_lim)
    sw_xticks = np.linspace(-shap_lim, shap_lim, 10)
    ax_sw.set_xticks(sw_xticks)
    ax_sw.set_xlabel("SHAP value (impact on model output)", fontsize=7)

    # y 轴标签映射
    display_feature_names = []
    unmapped = []
    for f in sorted_features:
        f_clean = str(f).strip()
        mapped = feature_name_map.get(f_clean)
        if mapped is None:
            mapped = feature_name_map.get(f_clean.replace("_lag", ""))
        if mapped is None:
            mapped = f_clean.replace("_lag", "")
            unmapped.append(f_clean)
        display_feature_names.append(mapped)

    ax_sw.set_yticks(y_pos)
    ax_sw.set_yticklabels(display_feature_names, fontsize=8)
    ax_sw.tick_params(axis="y", length=4)

    if feature_name_map and unmapped:
        print("❗未映射英文名：", unmapped)

    # bar 内标注
    for i, (v, p) in enumerate(zip(mean_shap_sorted, mean_shap_percent)):
        ax_bar.text(
            x=0.01 * xlim_bar,
            y=y_pos[i],
            s=f"{v:.4f} ({p:.2f}%)",
            va="center",
            ha="left",
            fontsize=8,
            color="black",
        )

    if out_path:
        plt.savefig(out_path, dpi=dpi, transparent=transparent, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

    return shap_df


# -----------------------------
# 2) RF 的 SHAP 计算
# -----------------------------
def _compute_shap_values_rf(model, X_eval, X_bg=None, positive_class=1, seed=None):
    """
    返回：正类(positive_class) 的 SHAP，强制输出 shape=(n_samples, n_features)
    兼容 shap 对 sklearn RF 的多种返回格式：
    - list: [class0_array, class1_array] 其中每个 shape=(n_samples,n_features)
    - ndarray: shape=(n_samples,n_features,n_classes)
    - ndarray: shape=(n_samples,n_features)
    """
    if X_bg is not None:
        explainer = shap.TreeExplainer(model, data=X_bg)
    else:
        explainer = shap.TreeExplainer(model)

    sv = explainer.shap_values(X_eval, check_additivity=False)

    # 情况1：list（常见）
    if isinstance(sv, list):
        sv_pos = sv[positive_class] if len(sv) > positive_class else sv[-1]
        sv_pos = np.asarray(sv_pos)
        if sv_pos.ndim == 3:
            k = sv_pos.shape[2]
            cls = positive_class if positive_class < k else (k - 1)
            sv_pos = sv_pos[:, :, cls]
        return sv_pos

    # 情况2：ndarray
    sv = np.asarray(sv)
    if sv.ndim == 3:
        k = sv.shape[2]
        cls = positive_class if positive_class < k else (k - 1)
        return sv[:, :, cls]
    if sv.ndim == 2:
        return sv

    raise ValueError(f"Unexpected shap_values ndim={sv.ndim}, shape={sv.shape}")

# -----------------------------
# 2.1) 抽样函数（你已有，保留）
# -----------------------------
def stratified_sample_eval_bg(
    test_df: pd.DataFrame,
    y_col: str,
    features: list[str],
    eval_n: int = 5000,
    bg_n: int = 1000,
    min_pos_eval: int = 200,
    min_pos_bg: int = 50,
    seed: int = 42,
):
    """
    分层/配额抽样：
    - eval：至少 min_pos_eval 个正类，其余由负类补足到 eval_n
    - bg：至少 min_pos_bg 个正类，其余由负类补足到 bg_n
    若正类不足，则“全取正类”并相应减少/调整配额。
    """
    rng = np.random.default_rng(seed)

    y = test_df[y_col].to_numpy()
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    if len(pos_idx) == 0:
        raise ValueError(f"test_df 中没有正类(=1)，无法做分层抽样：{y_col}")

    # ---------- eval ----------
    eval_n = min(eval_n, len(test_df))
    k_pos = min(len(pos_idx), min_pos_eval, eval_n)
    k_neg = eval_n - k_pos
    k_neg = min(k_neg, len(neg_idx))

    eval_pos = rng.choice(pos_idx, size=k_pos, replace=False) if k_pos > 0 else np.array([], dtype=int)
    eval_neg = rng.choice(neg_idx, size=k_neg, replace=False) if k_neg > 0 else np.array([], dtype=int)
    eval_idx = np.concatenate([eval_pos, eval_neg])
    rng.shuffle(eval_idx)

    # ---------- bg ----------
    bg_n = min(bg_n, len(test_df))
    b_pos = min(len(pos_idx), min_pos_bg, bg_n)
    b_neg = bg_n - b_pos
    b_neg = min(b_neg, len(neg_idx))

    bg_pos = rng.choice(pos_idx, size=b_pos, replace=False) if b_pos > 0 else np.array([], dtype=int)
    bg_neg = rng.choice(neg_idx, size=b_neg, replace=False) if b_neg > 0 else np.array([], dtype=int)
    bg_idx = np.concatenate([bg_pos, bg_neg])
    rng.shuffle(bg_idx)

    X_test = test_df[features]
    X_eval = X_test.iloc[eval_idx].copy()
    X_bg   = X_test.iloc[bg_idx].copy()

    # 供你打印核对用
    y_eval = y[eval_idx]
    y_bg   = y[bg_idx]

    return X_eval, X_bg, eval_idx, bg_idx, int((y_eval == 1).sum()), int((y_bg == 1).sum())

def allpos_sample_eval_bg(
    test_df: pd.DataFrame,
    y_col: str,
    features: list[str],
    neg_multiplier: int = 10,
    bg_n: int = 1000,
    pos_bg_max: int = 50,
    seed: int = 42,
):
    """
    Eval：全取正类 + neg_multiplier 倍负类
    Background：bg_n 中最多 pos_bg_max 个正类，其余负类
    """
    rng = np.random.default_rng(seed)

    y = test_df[y_col].to_numpy()
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    if len(pos_idx) == 0:
        raise ValueError("test 中没有正类(=1)，无法执行 all_pos 抽样。")

    # -------- Eval --------
    n_pos = len(pos_idx)
    n_neg_eval = min(len(neg_idx), neg_multiplier * n_pos)

    neg_eval_idx = rng.choice(neg_idx, size=n_neg_eval, replace=False)
    eval_idx = np.concatenate([pos_idx, neg_eval_idx])
    rng.shuffle(eval_idx)

    # -------- Background --------
    bg_n = min(bg_n, len(test_df))
    n_pos_bg = min(len(pos_idx), pos_bg_max, bg_n)
    n_neg_bg = min(len(neg_idx), bg_n - n_pos_bg)

    pos_bg_idx = rng.choice(pos_idx, size=n_pos_bg, replace=False)
    neg_bg_idx = rng.choice(neg_idx, size=n_neg_bg, replace=False)

    bg_idx = np.concatenate([pos_bg_idx, neg_bg_idx])
    rng.shuffle(bg_idx)

    X_test = test_df[features]
    X_eval = X_test.iloc[eval_idx].copy()
    X_bg   = X_test.iloc[bg_idx].copy()

    return X_eval, X_bg, len(pos_idx), n_neg_eval, n_pos_bg

# -----------------------------
# 3) 单任务：加载 RF → 抽样 → 计算 SHAP → 画图保存
# -----------------------------
def one_task_make_shap_fig_rf_allpos(
    set_name, ratio,
    data_bundles, features,
    model_dir, out_root,
    top_n=10,
    shap_eval_n=None,          # 保留参数位（当前 allpos 按 neg_multiplier 控制 eval 规模）
    shap_bg_n=1000,
    neg_multiplier=10,
    pos_bg_max=50,
    seed=42,
    feature_name_map: dict | None = None,
    y_col: str = "insider_trading",
):
    # --- paths ---
    model_dir = "/root/autodl-tmp/Table6/models"
    model_path = os.path.join(model_dir, f"rf_unified_{set_name}_ratio{ratio}.joblib")
    out_dir = os.path.join(out_root, set_name, f"ratio{ratio}")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"shap_bar_beeswarm_top{int(top_n)}.png")
    test_df = data_bundles[set_name]["test"]

    # seed 混入 set 和 ratio，保证不同任务抽样不同但可复现
    set_hash = abs(hash(set_name)) % 100000
    seed2 = int(seed) + int(ratio) + set_hash

    X_eval, X_bg, n_pos, n_neg_eval, n_pos_bg = allpos_sample_eval_bg(
        test_df=test_df,
        y_col=y_col,
        features=features,
        neg_multiplier=int(neg_multiplier),
        bg_n=int(shap_bg_n),
        pos_bg_max=int(pos_bg_max),
        seed=seed2,
    )

    X_eval = force_numeric_float32(X_eval)
    X_bg   = force_numeric_float32(X_bg)

    print(f"[{set_name} ratio={ratio}] pos={n_pos}, neg_eval={n_neg_eval}, bg_pos={n_pos_bg}")

    # --- load model ---
    model = joblib.load(model_path)

    # --- compute shap（正类=1）---
    shap_values = _compute_shap_values_rf(
        model=model,
        X_eval=X_eval.values,
        X_bg=X_bg.values,
        positive_class=1
    )

    # ✅ CHG: 接住 shap_df，用于统计 TopN 变量（频率表）
    shap_df = draw_shap_bar_beeswarm_onefig(
        shap_values=shap_values,
        X_shap=X_eval,
        feature_name_map=feature_name_map,
        top_n=int(top_n),
        out_path=out_path,
        dpi=300,
        transparent=True
    )

    top_features = shap_df["feature"].head(int(top_n)).tolist()

    return {
        "set": set_name,
        "ratio": ratio,
        "out_path": out_path,
        "pos_eval": int(n_pos),
        "neg_eval": int(n_neg_eval),
        "bg_pos": int(n_pos_bg),
        "sampling": "all_pos",
        "top_n": int(top_n),
        "top_features": top_features,
    }


# -----------------------------
# 4) 并行调度
# -----------------------------
def run_shap_figs_parallel_rf(
    sets, ratios,
    data_bundles, features,
    model_dir, out_root,
    top_n=10,
    shap_eval_n=5000,          # 保留（当前 allpos 不用 eval_n 但不影响调用）
    shap_bg_n=1000,
    seed=42,
    feature_name_map: dict | None = None,
    n_jobs=15
):
    tasks = [(s, r) for s in sets for r in ratios]

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
        delayed(one_task_make_shap_fig_rf_allpos)(
            s, r,
            data_bundles, features,
            model_dir, out_root,
            top_n=top_n,
            shap_eval_n=shap_eval_n,
            shap_bg_n=shap_bg_n,
            seed=seed,
            feature_name_map=feature_name_map,
        )
        for s, r in tasks
    )
    return pd.DataFrame(results)


# =============================
# ✅ CHG: TopN 变量频率统计工具
# =============================
def summarize_top_feature_frequency(df_res: pd.DataFrame, top_n: int, out_dir: str,
                                    fname_prefix: str = "top_features_frequency"):
    """
    汇总 df_res 中每个任务返回的 top_features，统计出现次数并输出频率表（CSV）。
    """
    if "top_features" not in df_res.columns:
        raise ValueError("df_res 中没有 'top_features' 列：请确认单任务函数返回了 top_features")

    counts = (
        df_res.explode("top_features")["top_features"]
        .dropna()
        .astype(str)
        .value_counts()
        .reset_index()
    )
    counts.columns = ["feature", "count_in_topk"]

    n_tasks = len(df_res)
    counts["share_of_tasks"] = counts["count_in_topk"] / n_tasks

    out_csv = os.path.join(out_dir, f"{fname_prefix}_top{int(top_n)}.csv")
    counts.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("Saved frequency table:", out_csv)

    return counts


# ============================================================
# ✅ MAIN：运行并输出频率表
# ============================================================

model_dir = "/root/autodl-tmp/Table6/models"
result_dir = "/root/autodl-tmp/Figure4"  # 顺便也定义一下
if __name__ == "__main__":
    # CHG: 这里的 sets/ratios 按你需求配置
    sets   = ["set1", "set2", "set3"]
    ratios = [1, 2, 5]

    # 运行：生成每个 set×ratio 的 SHAP 图，并返回每个任务的 TopN 特征
    df_res = run_shap_figs_parallel_rf(
        sets=sets, ratios=ratios,
        data_bundles=data_bundles,
        features=features,
        model_dir=model_dir,
        out_root=result_dir,
        top_n=10,
        shap_bg_n=1000,
        n_jobs=15
    )

    print(df_res[["set", "ratio", "out_path", "top_n"]].head())

    # CHG: 汇总 TopN 出现频次，保存到 result_dir
    freq_table = summarize_top_feature_frequency(
        df_res=df_res,
        top_n=int(df_res["top_n"].iloc[0]),
        out_dir=result_dir,
        fname_prefix="top_features_frequency"
    )

    print("\nTop frequency rows:\n", freq_table.head(30))

    top3 = freq_table.head(3)
    print("\nTop3:\n", top3)

[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.
 63%|=============       | 11533/18436 [07:06<04:14]       

[set1 ratio=1] pos=182, neg_eval=1820, bg_pos=50


 55%|===========         | 10191/18436 [07:11<05:48]       

[set1 ratio=2] pos=182, neg_eval=1820, bg_pos=50


 66%|=============       | 12161/18436 [07:35<03:54]       

...

[set2 ratio=2] pos=263, neg_eval=2630, bg_pos=50


 77%|===============     | 14229/18436 [09:56<02:56]       

[set2 ratio=5] pos=263, neg_eval=2630, bg_pos=50


100%|===================| 18419/18436 [12:48<00:00]       [Parallel(n_jobs=15)]: Done   9 out of   9 | elapsed: 13.7min finished


    set  ratio                                           out_path  top_n
0  set1      1  /root/autodl-tmp/Figure4/set1/ratio1/shap_bar_...     10
1  set1      2  /root/autodl-tmp/Figure4/set1/ratio2/shap_bar_...     10
2  set1      5  /root/autodl-tmp/Figure4/set1/ratio5/shap_bar_...     10
3  set2      1  /root/autodl-tmp/Figure4/set2/ratio1/shap_bar_...     10
4  set2      2  /root/autodl-tmp/Figure4/set2/ratio2/shap_bar_...     10
Saved frequency table: /root/autodl-tmp/Figure4/top_features_frequency_top10.csv

Top frequency rows:
                                               feature  count_in_topk  \
0                                          GDP_growth              8   
1      Beginning Balance of Cash and Cash Equivalents              6   
2                 Cash Dividend per Share after Taxes              6   
3             Cash Paid to and on Behalf of Employees              6   
4                Cash Dividend per Share before Taxes              6   
5                          